# Pipeline ML E2E con Prefect
Notebook actualizado para la tarea. Usa los módulos de la carpeta `Codigos`.

In [35]:
# Solo en Google Colab, descomentar si aplica:
# from google.colab import drive
# drive.mount('/content/drive')

!pip install -q prefect optuna mlflow xgboost streamlit

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")

print("Existe BASE_DIR:", BASE_DIR.exists())
print("Contenido de BASE_DIR:")
for item in BASE_DIR.iterdir():
    print("-", item.name)

Existe BASE_DIR: True
Contenido de BASE_DIR:
- README.md
- requirements.txt
- main_prefect.py
- dashboard.py
- Codigos
- Datos
- 7_Pipeline_prefect_actualizado.ipynb


In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")
CODE_DIR = BASE_DIR / "Codigos"

# Renombrar archivos con nombre incorrecto
if (CODE_DIR / "preprocesing.py").exists():
    (CODE_DIR / "preprocesing.py").rename(CODE_DIR / "preprocessing.py")

if (CODE_DIR / "postprocesing.py").exists():
    (CODE_DIR / "postprocesing.py").rename(CODE_DIR / "postprocessing.py")

if (CODE_DIR / "posprocesing.py").exists():
    (CODE_DIR / "posprocesing.py").rename(CODE_DIR / "posprocessing.py")

print("Archivos actuales en Codigos:")
for item in CODE_DIR.iterdir():
    print("-", item.name)

Archivos actuales en Codigos:
- preprocessing.py
- postprocessing.py
- monitoring.py
- inference.py
- training.py
- posprocessing.py
- __pycache__


In [ ]:
import sys
import importlib.util
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")
CODE_DIR = BASE_DIR / "Codigos"

def cargar_modulo(nombre_modulo, archivo_py):
    ruta = CODE_DIR / archivo_py
    print(f"Cargando {nombre_modulo} desde:", ruta)

    if not ruta.exists():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    spec = importlib.util.spec_from_file_location(nombre_modulo, ruta)
    modulo = importlib.util.module_from_spec(spec)

    # Registrar para que otros archivos puedan importarlo
    sys.modules[nombre_modulo] = modulo

    spec.loader.exec_module(modulo)
    return modulo

prep = cargar_modulo("preprocessing", "preprocessing.py")
aml = cargar_modulo("training", "training.py")
inf = cargar_modulo("inference", "inference.py")
mon = cargar_modulo("monitoring", "monitoring.py")
posp = cargar_modulo("postprocessing", "postprocessing.py")

print("Módulos cargados correctamente")

Cargando preprocessing desde: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos/preprocessing.py
Cargando training desde: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos/training.py
Cargando inference desde: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos/inference.py
Cargando monitoring desde: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos/monitoring.py
Cargando postprocessing desde: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos/postprocessing.py
Módulos cargados correctamente


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

try:
    from prefect import flow, task
except ImportError as exc:
    raise ImportError("Prefect no está instalado. Ejecutar: pip install prefect") from exc


BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")
CODE_DIR = BASE_DIR / "Codigos"

sys.path.insert(0, str(CODE_DIR))

import preprocessing as prep
import training as aml
import inference as inf
import monitoring as mon
import postprocessing as posp

print("BASE_DIR:", BASE_DIR)
print("CODE_DIR:", CODE_DIR)
print("Existe CODE_DIR:", CODE_DIR.exists())

BASE_DIR: /content/drive/MyDrive/TAREA PIPELINE ML E2E
CODE_DIR: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos
Existe CODE_DIR: True


In [ ]:
!pip install -r "/content/drive/MyDrive/TAREA PIPELINE ML E2E/requirements.txt"

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")

print("Archivos de entrenamiento:")
for archivo in (BASE_DIR / "Datos" / "Data Entrenamiento").iterdir():
    print("-", archivo.name)

print("\nArchivos de data cruda:")
for archivo in (BASE_DIR / "Datos" / "Data Cruda").iterdir():
    print("-", archivo.name)

Archivos de entrenamiento:
- Copia de p4_extrac.csv
- Copia de p2_extrac.csv
- Copia de p1_extrac.csv

Archivos de data cruda:
- p2_extrac.csv
- p10_extrac.csv


In [ ]:
payload = {
    "base_dir": str(BASE_DIR),
    "dir_codigos": str(BASE_DIR / "Codigos"),

    "DIR_TRAIN_RAWDATA": str(BASE_DIR / "Datos" / "Data Entrenamiento"),
    "DIR_RAWDATA": str(BASE_DIR / "Datos" / "Data Cruda"),
    "DIR_PROCESSED": str(BASE_DIR / "Datos" / "Data Preprocesada"),
    "TRAINING_DATA": str(BASE_DIR / "Datos" / "Data Preprocesada" / "training_data" / "preprocessed"),
    "MODEL_DIR": str(BASE_DIR / "Datos" / "Best_model"),
    "SCORE_DIR": str(BASE_DIR / "Datos" / "Output" / "score"),
    "MONITORING_DIR": str(BASE_DIR / "Datos" / "Output" / "monitoring"),
    "DIR_OUTPUT": str(BASE_DIR / "Datos" / "Output" / "final"),
    "REPLICA_DIR": str(BASE_DIR / "Datos" / "Output" / "replica"),

    "params": {
        "model_name": "extrac",
        "mode_type": "training",
        "partition": 10,
        "prepare_training_data": True,
        "auto_retrain": True,
        "psi_threshold": 0.25,
        "n_trials": 3
    }
}

payload

{'base_dir': '/content/drive/MyDrive/TAREA PIPELINE ML E2E',
 'dir_codigos': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Codigos',
 'DIR_TRAIN_RAWDATA': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Entrenamiento',
 'DIR_RAWDATA': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Cruda',
 'DIR_PROCESSED': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada',
 'TRAINING_DATA': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/training_data/preprocessed',
 'MODEL_DIR': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Best_model',
 'SCORE_DIR': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/score',
 'MONITORING_DIR': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/monitoring',
 'DIR_OUTPUT': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final',
 'REPLICA_DIR': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/replica',
 'params': {'model_name': 'extrac',
  'mode_type': 'training',
  '

In [ ]:
from prefect import flow, task
import os

@task
def preprocess_data(payload: dict):
    print("Iniciando preprocesamiento...")

    # 1. Preparar data de entrenamiento
    if payload["params"].get("prepare_training_data", True):
        prep.main(
            model_name=payload["params"]["model_name"],
            DIR_RAWDATA=payload["DIR_TRAIN_RAWDATA"],
            DIR_PROCESSED=payload["DIR_PROCESSED"],
            type_work="training",
            period=""
        )

    # 2. Preparar data de inferencia / OOT
    outputs = prep.main(
        model_name=payload["params"]["model_name"],
        DIR_RAWDATA=payload["DIR_RAWDATA"],
        DIR_PROCESSED=payload["DIR_PROCESSED"],
        type_work="inference",
        period=payload["params"]["partition"]
    )

    return outputs["vars"], outputs["post"]


@task
def train_model(payload: dict):
    print("Iniciando entrenamiento...")

    model_name = payload["params"]["model_name"]

    train_path = os.path.join(
        payload["DIR_PROCESSED"],
        "training_data",
        "preprocessed",
        f"train_vars_{model_name}.csv"
    )

    test_path = os.path.join(
        payload["DIR_PROCESSED"],
        "training_data",
        "preprocessed",
        f"test_vars_{model_name}.csv"
    )

    return aml.main(
        train_path=train_path,
        test_path=test_path,
        model_save_dir=payload["MODEL_DIR"],
        n_trials=payload["params"].get("n_trials", 3)
    )


@task
def run_inference(payload: dict, dir_pre_processed: str):
    print("Iniciando inferencia...")

    return inf.main(
        payload["MODEL_DIR"],
        dir_pre_processed,
        payload["SCORE_DIR"]
    )


@task
def run_monitoring(payload: dict, score_file_path: str):
    print("Iniciando monitoreo...")

    return mon.main(
        payload=payload,
        score_path=score_file_path
    )


@task
def run_postprocessing(payload: dict, dir_pos_processed: str, score_file_path: str):
    print("Iniciando postprocesamiento...")

    return posp.main(
        dir_pos_processed,
        score_file_path,
        payload["DIR_OUTPUT"]
    )


@flow(name="mlops_pipeline_cu_venta")
def mlops_pipeline(payload: dict):
    print("Ejecutando pipeline ML E2E...")

    # 1. Preprocesamiento
    dir_pre_processed, dir_pos_processed = preprocess_data(payload)

    # 2. Entrenamiento
    if payload["params"]["mode_type"] == "training":
        train_model(payload)

    # 3. Inferencia
    score_file_path = run_inference(payload, dir_pre_processed)

    # 4. Monitoreo
monitoring_result = run_monitoring(payload, score_file_path)

# 4.1 Reentrenamiento automático si se detecta deriva
if monitoring_result.get("requires_retraining", False):
    print("Alerta de monitoreo detectada. Se ejecutará reentrenamiento automático...")

    train_model(payload)

    print("Reentrenamiento finalizado. Se vuelve a ejecutar inferencia...")
    score_file_path = run_inference(payload, dir_pre_processed)

    print("Se vuelve a ejecutar monitoreo con el nuevo modelo...")
    monitoring_result = run_monitoring(payload, score_file_path)

# 5. Postprocesamiento
final_output_path = run_postprocessing(
    payload,
    dir_pos_processed,
    score_file_path
)

    return {
        "score_file_path": score_file_path,
        "monitoring_result": monitoring_result,
        "final_output_path": final_output_path
    }

print("Flujo mlops_pipeline definido correctamente")

Flujo mlops_pipeline definido correctamente


In [ ]:
payload["params"]["mode_type"] = "inference"
payload["params"]["prepare_training_data"] = False

In [ ]:
resultado = mlops_pipeline(payload)
resultado

INFO:prefect.flow_runs:Beginning flow run 'metal-falcon' for flow 'mlops_pipeline_cu_venta'


Ejecutando pipeline ML E2E...
Iniciando preprocesamiento...
Leyendo: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Entrenamiento/Copia de p1_extrac.csv
Leyendo: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Entrenamiento/Copia de p2_extrac.csv
Leyendo: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Entrenamiento/Copia de p4_extrac.csv
Data leída: 962,625 filas x 69 columnas
Archivos generados:
 - train_vars: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/training_data/preprocessed/train_vars_extrac.csv
 - test_vars: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/training_data/preprocessed/test_vars_extrac.csv
 - train_post: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/training_data/postprocessed/train_post_extrac.csv
 - test_post: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/training_data/postprocessed/test_post_extrac.csv
Leyendo: /content/drive/MyDrive/TAREA PIPELI

INFO:prefect.task_runs:Finished in state Completed()


Archivos generados:
 - vars: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/preprocessed/vars_10_extrac.csv
 - post: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Data Preprocesada/postprocessed/post_10_extrac.csv
Iniciando entrenamiento...


[I 2026-05-12 16:15:11,259] A new study created in memory with name: no-name-26dddca8-9d11-450d-898b-0dfd2a113089


...

PSI Score: 0.0039 | Flag: OK | Reentrenar: False
AUC OOT: 0.8611
Métricas guardadas en: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/monitoring/monitoring_metrics.json
Iniciando postprocesamiento...
Output TLV guardado en: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final/output_tlv_extrac_10.csv
Réplica guardada en: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final/replica/s3/scr_ec_omnicanal_10.txt
Réplica guardada en: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final/replica/athena/scr_ec_omnicanal_10.txt


INFO:prefect.task_runs:Finished in state Completed()


Réplica guardada en: /content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final/replica/onpremise/scr_ec_omnicanal_10.txt


INFO:prefect.flow_runs:Finished in state Completed()


{'score_file_path': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/score/inference_extrac_10.csv',
 'monitoring_result': {'psi_score': 0.0038831188806607905,
  'psi_flag': 'OK',
  'requires_retraining': False,
  'auc': 0.8610762948360515,
  'recall_at_50': 0.0012447099825740602},
 'final_output_path': '/content/drive/MyDrive/TAREA PIPELINE ML E2E/Datos/Output/final/output_tlv_extrac_10.csv'}

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TAREA PIPELINE ML E2E")

output_tlv = BASE_DIR / "Datos" / "Output" / "final" / "output_tlv_extrac_10.csv"
monitoring = BASE_DIR / "Datos" / "Output" / "monitoring" / "monitoring_metrics.json"

print("Existe output TLV:", output_tlv.exists())
print("Existe monitoring:", monitoring.exists())

Existe output TLV: True
Existe monitoring: True


In [ ]:
!pip install streamlit pyngrok